# Walmart Weekly Sales Forecasting
### Data Preprocessing

In [ ]:
import pandas as pd

# Load the data
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
stores = pd.read_csv("../data/stores.csv")
features = pd.read_csv("../data/features.csv")

# Create main table with left join (first with "stores", then with "features")
data = pd.merge(train, stores, on="Store", how="left")
data = pd.merge(data, features, on=["Store", "Date"], how="left")

# There are identical IsHoliday columns (IsHoldiay_x, IsHoliday_y), delete one of them and rename the other one
data = data.drop(columns="IsHoliday_y")
data = data.rename(columns={"IsHoliday_x": "IsHoliday"})

# There are blanks in the markdown columns. NaN values will be 0.
markdown_columns = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
data[markdown_columns] = data[markdown_columns].fillna(0)

### Feature Engineering

In [ ]:
# Date type: string -> datetime
data['Date'] = pd.to_datetime(data['Date'])
# New features: Year, Month, Week of Year
data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month
data['Week_of_Year'] = data['Date'].dt.isocalendar().week.astype(int)
data = data.drop(columns=['Date'])

# IsHoliday sütununu 0 ve 1'lere dönüştürüyoruz
data['IsHoliday'] = data['IsHoliday'].astype(int)

# Type sütununu One-Hot Encoding ile dönüştürüyoruz: A-B-C -> 
data = pd.get_dummies(data, columns=['Type'], drop_first=True, dtype=int)
# drop first: sadece B ve C sütununu oluşturur ve eğer ikisi sıfırsa model A olduğu çıkarımını yapar. Daha az sütunla sorun çözülür.

# Last Week Sales sütununu oluşturma
data = data.sort_values(by=['Store', 'Dept', 'Date']).reset_index(drop=True)
data['Last_Week_Sales'] = data.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(1)
data['Last_Week_Sales'] = data['Last_Week_Sales'].fillna(0)

# Tahmin etmek istedğimiz, target
y = data['Weekly_Sales']

# Modelin kullanacağı öznitelikler
X = data.drop(columns=['Weekly_Sales'])

### Data Splitting

In [ ]:
split_point = int(len(X) * 0.8)
X_train, X_val = X.iloc[:split_point], X.iloc[split_point:]
y_train, y_val = y.iloc[:split_point], y.iloc[split_point:]

### Model Import

In [ ]:
import xgboost as xgb

main_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1 # To use all of the cpu cores
)

### Feature Selection

In [ ]:
# RFE Selection
from sklearn.feature_selection import RFE
rfe_selector = RFE(estimator=main_model, n_features_to_select=10)
rfe_selector.fit(X_train, y_train)
rfe_selected_columns = X_train.columns[rfe_selector.support_].tolist()

# MRMR Selection
from mrmr import mrmr_regression
mrmr_selected_columns = mrmr_regression(X=X_train, y=y_train, K=10, show_progress=False) # Best 10

### Model Training and Prediction

In [ ]:
# MRMR
main_model.fit(X_train[mrmr_selected_columns], y_train)
prediction_mrmr = main_model.predict(X_val[mrmr_selected_columns])

# RFE
main_model.fit(X_train[rfe_selected_columns], y_train)
prediction_rfe = main_model.predict(X_val[rfe_selected_columns])

# All Features
main_model.fit(X_train, y_train)
prediction_all = main_model.predict(X_val)

### Model Validation and Results

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

def wmape(real, prediction):
    return (np.abs(real - prediction).sum() / real.sum()) * 100

results = {
    "MRMR Selection": {
        "RMSE ($)": np.sqrt(mean_squared_error(y_val, prediction_mrmr)),
        "WMAPE (%)": wmape(y_val.values, prediction_mrmr)
    },
    "RFE Selection": {
        "RMSE ($)": np.sqrt(mean_squared_error(y_val, prediction_rfe)),
        "WMAPE (%)": wmape(y_val.values, prediction_rfe)
    },
    "All Features": {
        "RMSE ($)": np.sqrt(mean_squared_error(y_val, prediction_all)),
        "WMAPE (%)": wmape(y_val.values, prediction_all)
    }
}

report_df = pd.DataFrame(results).T.round(2)
report_df